# Kỹ thuật 2: Ngưỡng thích nghi bằng ảnh tích phân

In [1]:
import os
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'

SEED = 2026

## 1. Xây dựng các hàm

In [2]:
def integral_image(img):
    # Ép sang int64 để tránh tràn số khi cộng dồn
    img_64 = img.astype(np.int64)
    H, W = img.shape
    
    # Ma trận tích phân kích thước (H+1, W+1), hàng 0 và cột 0 bằng 0
    S = np.zeros((H + 1, W + 1), dtype=np.int64)
    S[1:, 1:] = np.cumsum(np.cumsum(img_64, axis=0), axis=1)
    return S


def box_sum(S, y1, x1, y2, x2):
    # Công thức hình học 4 đỉnh: D - B - C + A
    return S[y2 + 1, x2 + 1] - S[y1, x2 + 1] - S[y2 + 1, x1] + S[y1, x1]


def local_mean_std(img, radius):
    H, W = img.shape
    img_64 = img.astype(np.int64)
    
    # Tính 2 ảnh tích phân cho I và I^2
    S1 = integral_image(img_64)
    S2 = integral_image(img_64 ** 2)

    # Tạo lưới tọa độ và kẹp biên cửa sổ
    ys = np.arange(H)[:, None]
    xs = np.arange(W)[None, :]
    y1 = np.maximum(ys - radius, 0)
    y2 = np.minimum(ys + radius, H - 1)
    x1 = np.maximum(xs - radius, 0)
    x2 = np.minimum(xs + radius, W - 1)
    
    # Số pixel thực tế trong từng cửa sổ
    count = ((y2 - y1 + 1) * (x2 - x1 + 1)).astype(np.float64)

    sum1 = box_sum(S1, y1, x1, y2, x2).astype(np.float64)
    sum2 = box_sum(S2, y1, x1, y2, x2).astype(np.float64)

    mean = sum1 / count
    var = sum2 / count - mean ** 2
    
    # Kẹp phương sai không âm để tránh lỗi sqrt ra NaN
    var = np.maximum(var, 0.0)
    std = np.sqrt(var)
    return mean, std


def adaptive_threshold(img, radius=15, method='bradley', t=0.15, k=0.34):
    mean, std = local_mean_std(img, radius)
    method = method.lower()
    
    if method == 'bradley':
        T = mean * (1.0 - t)
    elif method == 'sauvola':
        R = 128.0
        T = mean * (1.0 + k * (std / R - 1.0))
    else:
        raise ValueError("Method phải là 'bradley' hoặc 'sauvola'")
        
    return np.where(img < T, 0, 255).astype(np.uint8)

## Kiểm tra thử các hàm

In [3]:
rng = np.random.default_rng(SEED)

# 1) Kiểm tra kích thước và viền 0
A = rng.integers(0, 256, size=(97, 131), dtype=np.uint8)
S = integral_image(A)
assert S.shape == (98, 132) and S.dtype == np.int64
assert not S[0, :].any() and not S[:, 0].any()
assert S[-1, -1] == int(A.astype(np.int64).sum())

# 2) Kiểm tra box_sum với 1000 vùng ngẫu nhiên
for _ in range(1000):
    y1, y2 = np.sort(rng.integers(0, A.shape[0], 2))
    x1, x2 = np.sort(rng.integers(0, A.shape[1], 2))
    assert box_sum(S, y1, x1, y2, x2) == int(A[y1:y2+1, x1:x2+1].astype(np.int64).sum())
assert box_sum(S, 0, 0, 0, 0) == A[0, 0]

# 3) Kiểm tra local_mean_std trên ảnh nhỏ
B = rng.integers(0, 256, size=(23, 29), dtype=np.uint8)
r = 4
m, s = local_mean_std(B, r)
for y in range(B.shape[0]):
    for x in range(B.shape[1]):
        win = B[max(y-r, 0):y+r+1, max(x-r, 0):x+r+1].astype(np.float64)
        assert abs(m[y, x] - win.mean()) < 1e-9
        assert abs(s[y, x] - win.std()) < 1e-6

# 4) Kiểm tra chống tràn số int32
big = np.full((2000, 2000), 255, dtype=np.uint8)
S2_big = integral_image(big.astype(np.int64) ** 2)
assert S2_big[-1, -1] == 255**2 * 2000 * 2000

# 5) Ảnh hằng: std phải đúng bằng 0, không bị NaN
const = np.full((50, 60), 200, dtype=np.uint8)
m_c, s_c = local_mean_std(const, 7)
assert np.allclose(m_c, 200) and not np.isnan(s_c).any() and (s_c == 0).all()

# 6) Đầu ra adaptive_threshold
out = adaptive_threshold(A, 7, 'sauvola')
assert out.dtype == np.uint8 and set(np.unique(out)) <= {0, 255}

print("Đã kiểm tra xong, 4 hàm hoạt động chính xác.")

Đã kiểm tra xong, 4 hàm hoạt động chính xác.
